# Graphs, roles, and verdicts

`CausalGraph` is a DAG as a `Spec` — own adjacency implementation, no networkx. Bidirected
edges (`A <-> B`) express latent confounding and are handled internally as hidden common
causes, so every query works on a semi-Markovian graph. `identify()` returns an honest
verdict: the route the graph licenses, the alternatives, and the assumptions each needs.

In [ ]:
from axiom.core import Spec
from axiom.identify import (
    CausalGraph, GraphError, IdentificationVerdict, Role, RoleAssignment, Route, adjustment_sets,
    admissible_set_exists, assign_roles, backdoor_admissible, canonical_adjustment_set, identify,
    minimal_adjustment_sets, parse_edges, requires_unmeasured, roles,
)

## Building a graph

`from_edges` parses a compact edge list. `unmeasured` marks nodes the panel does not observe.

In [ ]:
g = CausalGraph.from_edges("Z -> X, Z -> Y, X -> M, M -> Y, X <-> W, W -> Y", unmeasured=["W"], name="toy")
print(g)
print(g.nodes, "| measured:", sorted(g.measured))
print(g.parents("Y"), g.children("X"), g.siblings("X"))
print(g.topological_order())
print(parse_edges("A -> B; C <- B"))

In [ ]:
try:
    CausalGraph.from_edges("A -> B, B -> C, C -> A")
except GraphError as e:
    print("refused:", e)

## d-separation and graph surgery

`d_separated(x, y, z)` runs Bayes-ball on the augmented DAG. `remove_edges_out_of(x)` is
$G_{\underline{X}}$ (the back-door graph); `remove_edges_into(x)` is $G_{\overline{X}}$.

In [ ]:
collider = CausalGraph.from_edges("A -> C, B -> C, C -> D")
print(collider.d_separated("A", "B"), collider.d_separated("A", "B", ["C"]), collider.d_separated("A", "B", ["D"]))
under = g.remove_edges_out_of("X")
print(under.d_separated("X", "Y", ["Z"]), "<- the latent path X <-> W -> Y stays open")

## Adjustment sets

`backdoor_admissible` is Pearl's back-door criterion. `adjustment_sets` enumerates every
admissible set among measured candidates; `minimal_adjustment_sets` the inclusion-minimal
ones; `canonical_adjustment_set` is the polynomial-time set of van der Zander et al. (2014),
and `admissible_set_exists` answers the existence question without enumerating.
`requires_unmeasured` is the downgrade signal: a set exists, but only with an unobserved node.

In [ ]:
simple = CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y")
print(backdoor_admissible(simple, "X", "Y", ["Z"]), backdoor_admissible(simple, "X", "Y", []))
print(adjustment_sets(simple, "X", "Y"), minimal_adjustment_sets(simple, "X", "Y"))
print(canonical_adjustment_set(simple, "X", "Y"), admissible_set_exists(simple, "X", "Y"))
print("needs an unmeasured node:", requires_unmeasured(g, "X", "Y"))

M-bias: conditioning on the collider `M` *opens* a path. The empty set is admissible; `{M}` is not.

In [ ]:
m_bias = CausalGraph.from_edges("U1 -> X, U1 -> M, U2 -> M, U2 -> Y, X -> Y", unmeasured=["U1", "U2"])
print(backdoor_admissible(m_bias, "X", "Y", []), backdoor_admissible(m_bias, "X", "Y", ["M"]))
print(minimal_adjustment_sets(m_bias, "X", "Y"))

## Roles

`roles` classifies every node relative to `x -> y` — confounder, mediator, collider, instrument,
proxy, descendant of the outcome, neutral — with a documented precedence. `assign_roles`
wraps it as a `RoleAssignment` spec carrying the graph's hash.

In [ ]:
r: dict[str, Role] = roles(m_bias, "X", "Y")
print(r)
ra: RoleAssignment = assign_roles(g, "X", "Y")
print(ra.roles, ra.graph_hash[:12])
print("confounders:", ra.with_role("confounder"), "| mediators:", ra.with_role("mediator"))

## The verdict

`identify()` tries the routes in preference order — back-door, front-door, instrument,
back-door-with-an-unmeasured-node — and reports the best *and* the alternatives. A measured
back-door or front-door set is `identified`. An instrument is `downgraded`: the graph gives
exclusion, but relevance is a data question and effect homogeneity (or monotonicity, for a
LATE) is not something a graph can establish. An adjustment set needing an unmeasured node is
`downgraded` with the node named, and nothing downstream reports a point estimate for it
without an explicit, ledgered `assume_identified=True` (Phase 4).

In [ ]:
v: IdentificationVerdict = identify(simple, "X", "Y")
print(v.status, v.route, v.adjustment_set, v.alternatives)

hidden = simple.with_unmeasured("Z")
v = identify(hidden, "X", "Y")
print(v.status, v.route, v.unmeasured_required, [a.name for a in v.verdict.assumptions])

iv = CausalGraph.from_edges("Z -> X, X -> Y, X <-> Y")
v = identify(iv, "X", "Y")
print(v.status, v.route, v.instrument, [f"{a.name}:{a.state}" for a in v.verdict.assumptions])

blocked = CausalGraph.from_edges("X -> Y, X <-> Y")
v = identify(blocked, "X", "Y")
print(v.status, v.route, "|", v.verdict.reason)

### Feedback (review B4)

A static summary graph can hide treatment–outcome feedback over time (doses respond to
yesterday's outcome). Declare it with `feedback=True`; when the treatment also has carryover,
the verdict is downgraded with `no_time_varying_confounding` unverified — static adjustment is
not valid there.

In [ ]:
fb = simple.model_copy(update={"feedback": True})
print(identify(fb, "X", "Y").status, identify(fb, "X", "Y", has_carryover=True).status)
print([a.name for a in identify(fb, "X", "Y", has_carryover=True).verdict.assumptions])

In [ ]:
pref: tuple[Route, ...] = ("frontdoor", "backdoor")
fd = CausalGraph.from_edges("X -> M, M -> Y, X <-> Y")
print(identify(fd, "X", "Y", prefer=pref).route)
print(Spec.from_json(v.to_json()) == v)